Sarah Sullivan

October 23, 2025

Last Updated: December 13, 2025

PSID 

In [4]:
# Import necessary packages
import pandas as pd
import numpy as np
import ast

In [5]:
# Define root
root = "/Users/sarsul/Library/CloudStorage/Dropbox-UniversityofMichigan/Sarah Sullivan/SARAH-SOFT/research/psid/"

In [6]:
df = pd.read_stata(root + "01_try_pyth_v13.dta")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/sarsul/Library/CloudStorage/Dropbox-UniversityofMichigan/Sarah Sullivan/SARAH-SOFT/research/psid/01_try_pyth_v13.dta'

In [33]:
df['hhr_prev'] = df['hhr_'].shift(1)
df['ages_prev'] = df['ages'].shift(1)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/4011252393.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['hhr_prev'] = df['hhr_'].shift(1)
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/4011252393.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ages_prev'] = df['ages'].shift(1)


In [34]:
def parse_tuple_string(x):
    if isinstance(x, str):
        try:
            result = ast.literal_eval(x)
            # Handle case where literal_eval returns a single value (int, float, etc.)
            if isinstance(result, (list, tuple)):
                return list(result)
            else:
                return [result]  # Wrap single values in a list
        except (SyntaxError, ValueError):
            return []
    elif isinstance(x, (list, tuple)):
        return list(x)
    else:
        # For other types (int, float, etc.), wrap in a list
        return [x] if x is not None else []

In [ ]:
df['hhr_prev'] = df['hhr_prev'].apply(parse_tuple_string)
df['hhr_'] = df['hhr_'].apply(parse_tuple_string)

In [37]:
df['who_left'] = df.apply(lambda row: [] if not row['hhr_prev'] else [x for x in row['hhr_prev'] if x not in row['hhr_']], axis=1)
df['who_came'] = df.apply(lambda row: [] if not row['hhr_prev'] else [x for x in row['hhr_'] if x not in row['hhr_prev']], axis=1)

In [ ]:
df['ages'] = df['ages'].apply(parse_tuple_string)
df['ages_prev'] = df['ages_prev'].apply(parse_tuple_string)

In [ ]:
def get_ages_left(row):
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    who_left = row['who_left'] if isinstance(row['who_left'], list) else []
    return [ages_prev[hhr_prev.index(pid)] for pid in who_left if pid in hhr_prev and hhr_prev.index(pid) < len(ages_prev)]

def get_ages_came(row):
    hhr_ = row['hhr_'] if isinstance(row['hhr_'], list) else []
    ages = row['ages'] if isinstance(row['ages'], list) else []
    who_came = row['who_came'] if isinstance(row['who_came'], list) else []
    return [ages[hhr_.index(pid)] for pid in who_came if pid in hhr_ and hhr_.index(pid) < len(ages)]

In [ ]:
df['ages_left'] = df.apply(get_ages_left, axis=1)
df['ages_came'] = df.apply(get_ages_came, axis=1)

In [46]:
df['adult_came'] = df['ages_came'].apply(
    lambda ages: int(isinstance(ages, list) and any(age >= 18 for age in ages))
)

df['child_came'] = df['ages_came'].apply(
    lambda ages: int(isinstance(ages, list) and any(age < 18 for age in ages))
)

df['adult_left'] = df['ages_left'].apply(
    lambda ages: int(isinstance(ages, list) and any(age >= 18 for age in ages))
)

df['child_left'] = df['ages_left'].apply(
    lambda ages: int(isinstance(ages, list) and any(age < 18 for age in ages))
)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/1250763495.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['adult_came'] = df['ages_came'].apply(
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/1250763495.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['child_came'] = df['ages_came'].apply(
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/1250763495.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many t

In [47]:
df['n_adults_left'] = df['ages_left'].apply(
    lambda ages: sum(age >= 18 for age in ages) if isinstance(ages, list) else 0
)

df['n_adults_came'] = df['ages_came'].apply(
    lambda ages: sum(age >= 18 for age in ages) if isinstance(ages, list) else 0
)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/4178870997.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['n_adults_left'] = df['ages_left'].apply(
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/4178870997.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['n_adults_came'] = df['ages_came'].apply(


In [48]:
# Create sib_list: list of all sibling IDs (ID_S01 through ID_S16)
sib_cols = [f'ID_S{str(i).zfill(2)}' for i in range(1, 17)]
df['sib_list'] = df[sib_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/1462560733.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['sib_list'] = df[sib_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)


In [49]:
df['sib_came'] = df.apply(
    lambda row: int(isinstance(row['who_came'], list) and isinstance(row['sib_list'], list) and any(id in row['sib_list'] for id in row['who_came'])),
    axis=1
)

df['sib_left'] = df.apply(
    lambda row: int(isinstance(row['who_left'], list) and isinstance(row['sib_list'], list) and any(id in row['sib_list'] for id in row['who_left'])),
    axis=1
)   

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/2983105489.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['sib_came'] = df.apply(
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/2983105489.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['sib_left'] = df.apply(
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/2983105489.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performan

In [50]:
# Get ages of siblings who came
def get_sib_ages_came(row):
    if not isinstance(row['who_came'], list) or not isinstance(row['sib_list'], list):
        return []
    hhr_ = row['hhr_'] if isinstance(row['hhr_'], list) else []
    ages = row['ages'] if isinstance(row['ages'], list) else []
    # Get IDs that are both in who_came and sib_list
    sibs_who_came = [id for id in row['who_came'] if id in row['sib_list']]
    # Get ages for those siblings
    return [ages[hhr_.index(sib_id)] for sib_id in sibs_who_came if sib_id in hhr_ and hhr_.index(sib_id) < len(ages)]

# Get ages of siblings who left
def get_sib_ages_left(row):
    if not isinstance(row['who_left'], list) or not isinstance(row['sib_list'], list):
        return []
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    # Get IDs that are both in who_left and sib_list
    sibs_who_left = [id for id in row['who_left'] if id in row['sib_list']]
    # Get ages for those siblings
    return [ages_prev[hhr_prev.index(sib_id)] for sib_id in sibs_who_left if sib_id in hhr_prev and hhr_prev.index(sib_id) < len(ages_prev)]

df['sib_ages_came'] = df.apply(get_sib_ages_came, axis=1)
df['sib_ages_left'] = df.apply(get_sib_ages_left, axis=1)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/2113840981.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['sib_ages_came'] = df.apply(get_sib_ages_came, axis=1)
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/2113840981.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['sib_ages_left'] = df.apply(get_sib_ages_left, axis=1)
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/2113840981.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the resu

In [52]:
# Create gpar_list: list of ID's for all grandparents
gpar_cols = ['ID_aM_aM', 'ID_aM_aD', 'ID_aM_bM', 'ID_aM_bD', 'ID_aD_aM', 'ID_aD_aD', 'ID_aD_bM', 'ID_aD_bD', 
             'ID_bM_aM', 'ID_bM_aD', 'ID_bM_bM', 'ID_bM_bD', 'ID_bD_aM', 'ID_bD_aD', 'ID_bD_bM', 'ID_bD_bD']
df['gpar_list'] = df[gpar_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/3932152440.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['gpar_list'] = df[gpar_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)


In [53]:
# Create par_list: list of ID's for parents
par_cols = ['ID_aM', 'ID_aD', 'ID_bM', 'ID_bD']
df['par_list'] = df[par_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/2283279806.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['par_list'] = df[par_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)


In [54]:
df['gpar_came'] = df.apply(
    lambda row: int(isinstance(row['who_came'], list) and isinstance(row['gpar_list'], list) and any(id in row['gpar_list'] for id in row['who_came'])),
    axis=1
)

df['gpar_left'] = df.apply(
    lambda row: int(isinstance(row['who_left'], list) and isinstance(row['gpar_list'], list) and any(id in row['gpar_list'] for id in row['who_left'])),
    axis=1
)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/2409024252.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['gpar_came'] = df.apply(
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/2409024252.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['gpar_left'] = df.apply(
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/2409024252.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform

In [55]:
df['par_came'] = df.apply(
    lambda row: int(isinstance(row['who_came'], list) and isinstance(row['par_list'], list) and any(id in row['par_list'] for id in row['who_came'])),
    axis=1
)

df['par_left'] = df.apply(
    lambda row: int(isinstance(row['who_left'], list) and isinstance(row['par_list'], list) and any(id in row['par_list'] for id in row['who_left'])),
    axis=1
)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/216790329.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['par_came'] = df.apply(
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/216790329.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['par_left'] = df.apply(
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/216790329.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.

In [58]:
# Create relatives list
rel_cols = sib_cols + par_cols + gpar_cols

In [59]:
df['rel_list'] = df[rel_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/542674756.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['rel_list'] = df[rel_cols].apply(lambda row: [x for x in row if pd.notna(x)], axis=1)


In [60]:
df['other_came'] = df.apply(
    lambda row: int(isinstance(row['who_came'], list) and isinstance(row['rel_list'], list) and any(id not in row['rel_list'] for id in row['who_came'])),
    axis=1
)

df['other_left'] = df.apply(
    lambda row: int(isinstance(row['who_left'], list) and isinstance(row['rel_list'], list) and any(id not in row['rel_list'] for id in row['who_left'])),
    axis=1
)

/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/728185986.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['other_came'] = df.apply(
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/728185986.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['other_left'] = df.apply(
/var/folders/xv/wbxpjr7n5w35wmbmy5tx74n40000gp/T/ipykernel_42879/728185986.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

In [63]:
# Set who_came and ages_came to empty lists for first observation of each ID
df['is_first_obs'] = df.groupby('ID')['yr'].rank(method='first') == 1
df.loc[df['is_first_obs'], 'who_came'] = df.loc[df['is_first_obs'], 'who_came'].apply(lambda x: [])
df.loc[df['is_first_obs'], 'ages_came'] = df.loc[df['is_first_obs'], 'ages_came'].apply(lambda x: [])

In [64]:
df.to_csv('final_data_v2.csv', index=False)